<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/06_statistical_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 6: RQ1-RQ4 Statistical Tests

Runs the exact tests from Table 1 of the Synopsis, each preceded by a
Shapiro-Wilk normality check with the specified non-parametric fallback.

- **RQ1:** One-sample t-test on CAR_short (H0: CAR = 0) — fallback: Wilcoxon signed-rank
- **RQ2:** One-way ANOVA across announcement_type — fallback: Kruskal-Wallis; Tukey HSD post-hoc if significant
- **RQ3:** Multiple regression with interaction (firm_size x announcement_type)
- **RQ4:** Independent-samples t-test, tech vs. non-tech — Levene's test for variance equality; Welch's correction if violated

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 162 (delta 59), reused 142 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 1.11 MiB | 6.80 MiB/s, done.
Resolving deltas: 100% (59/59), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas numpy scipy statsmodels scikit-learn

## Cell 3 — Configuration

In [ ]:
import os

DATA_FILE = os.path.join(BASE_DIR, "data/processed/analysis_dataset.csv")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ALPHA = 0.05

import pandas as pd
df = pd.read_csv(DATA_FILE)
print(f"Loaded {len(df)} events from {DATA_FILE}")
df.head()

Loaded 51 events from /content/QM640-WALSH-CAPSTONE/data/processed/analysis_dataset.csv


,event_id,ticker,event_date,announcement_type,firm_size_log,sector,alpha,beta,CAR_short,CAR_long,sector_binary
0,0001493152-24-039310:ex99-1.htm,SLNHP,2024-10-03,M&A,NaN,NaN,0.006617,0.942315,-0.170204,0.665825,Non-Technology
1,0001213900-25-119878:ea026910601ex99-1_healthc...,HCTI,2025-12-10,M&A,NaN,NaN,0.001070,5.799071,-0.054362,-1.974139,Non-Technology
2,0000796343-24-000057:adbeex991q124.htm,ADBE,2024-03-14,R&D,25.217295,Information Technology,0.000505,1.555008,-0.089039,-0.165778,Technology
3,0001213900-25-086243:ea025660401ex99-1_sound.htm,SOUN,2025-09-09,M&A,NaN,NaN,-0.000998,2.841992,-0.075929,0.172192,Non-Technology
4,0001437749-24-030748:ex_730179.htm,LODE,2024-10-07,M&A,NaN,NaN,-0.006642,1.079106,-0.090387,-0.193151,Non-Technology


## RQ1 — Does CAR differ significantly from zero?

In [ ]:
from scipy import stats

def rq1_test(df):
    car = df["CAR_short"].dropna()
    _, p_norm = stats.shapiro(car)
    print(f"Shapiro-Wilk normality test: p = {p_norm:.4f}")

    if p_norm >= ALPHA:
        t_stat, p_val = stats.ttest_1samp(car, 0)
        print(f"One-sample t-test: t = {t_stat:.3f}, p = {p_val:.4f}")
        method = "one-sample t-test"
    else:
        stat, p_val = stats.wilcoxon(car)
        t_stat = stat
        print(f"Normality violated -> Wilcoxon signed-rank test: W = {stat:.3f}, p = {p_val:.4f}")
        method = "Wilcoxon signed-rank (non-parametric fallback)"

    caar = car.mean()
    print(f"CAAR (mean CAR): {caar:.4%}")
    print(f"Result: {'REJECT' if p_val < ALPHA else 'FAIL TO REJECT'} H0 at alpha = .05")

    return {"RQ": "RQ1", "method": method, "statistic": t_stat, "p_value": p_val,
            "n": len(car), "CAAR": caar}


rq1_result = rq1_test(df)

Shapiro-Wilk normality test: p = 0.0000
Normality violated -> Wilcoxon signed-rank test: W = 466.000, p = 0.0648
CAAR (mean CAR): -0.1632%
Result: FAIL TO REJECT H0 at alpha = .05


## RQ2 — Does CAR differ by announcement type?

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def rq2_test(df):
    groups = [g["CAR_short"].dropna() for _, g in df.groupby("announcement_type")]

    normal = all(stats.shapiro(g)[1] >= ALPHA for g in groups if len(g) >= 3)
    _, p_levene = stats.levene(*groups)
    print(f"Normality (all groups): {'OK' if normal else 'violated'} | "
          f"Levene's test (equal variance): p = {p_levene:.4f}")

    if normal:
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"One-way ANOVA: F = {f_stat:.3f}, p = {p_val:.4f}")
        method = "one-way ANOVA"
        if p_val < ALPHA:
            tukey = pairwise_tukeyhsd(df["CAR_short"].dropna(),
                                       df.loc[df["CAR_short"].notna(), "announcement_type"])
            print("\nTukey HSD post-hoc comparisons:")
            print(tukey)
    else:
        h_stat, p_val = stats.kruskal(*groups)
        f_stat = h_stat
        print(f"Normality violated -> Kruskal-Wallis: H = {h_stat:.3f}, p = {p_val:.4f}")
        method = "Kruskal-Wallis (non-parametric fallback)"

    ss_between = sum(len(g) * (g.mean() - df["CAR_short"].mean()) ** 2 for g in groups)
    ss_total = ((df["CAR_short"].dropna() - df["CAR_short"].mean()) ** 2).sum()
    eta_sq = ss_between / ss_total if ss_total else float("nan")
    print(f"Effect size (eta-squared): {eta_sq:.4f}")

    return {"RQ": "RQ2", "method": method, "statistic": f_stat, "p_value": p_val,
            "n": len(df), "eta_squared": eta_sq}


rq2_result = rq2_test(df)

Normality (all groups): violated | Levene's test (equal variance): p = 0.8296
Normality violated -> Kruskal-Wallis: H = 0.552, p = 0.7590
Effect size (eta-squared): 0.0018


## RQ3 — Does firm size moderate announcement-type -> CAR?

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

def rq3_test(df):
    model_df = df.dropna(subset=["CAR_short", "firm_size_log", "announcement_type"]).copy()
    model = smf.ols("CAR_short ~ firm_size_log * C(announcement_type)", data=model_df).fit()
    print(model.summary())

    X = sm.add_constant(pd.get_dummies(
        model_df[["firm_size_log", "announcement_type"]], drop_first=True
    ).astype(float))
    vif = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    print("\nVariance Inflation Factors:")
    print(vif)

    return {"RQ": "RQ3", "method": "multiple regression with interaction",
            "statistic": model.fvalue, "p_value": model.f_pvalue,
            "n": len(model_df), "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj}


rq3_result = rq3_test(df)

                            OLS Regression Results                            
Dep. Variable:              CAR_short   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Sun, 26 Jul 2026   Prob (F-statistic):                nan
Time:                        10:54:24   Log-Likelihood:                 108.37
No. Observations:                   3   AIC:                            -210.7
Df Residuals:                       0   BIC:                            -213.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 3 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  

## RQ4 — Does CAR differ between tech and non-tech sectors?

In [ ]:
def rq4_test(df):
    tech = df[df["sector_binary"] == "Technology"]["CAR_short"].dropna()
    nontech = df[df["sector_binary"] == "Non-Technology"]["CAR_short"].dropna()

    _, p_levene = stats.levene(tech, nontech)
    equal_var = p_levene >= ALPHA
    variance_note = "equal variance assumed" if equal_var else "Welch's correction applied"
    print(f"Levene's test (equal variance): p = {p_levene:.4f} -> {variance_note}")

    t_stat, p_val = stats.ttest_ind(tech, nontech, equal_var=equal_var)
    mean_diff = tech.mean() - nontech.mean()
    print(f"Independent-samples t-test: t = {t_stat:.3f}, p = {p_val:.4f}")
    print(f"Mean CAR difference (tech - non-tech): {mean_diff:.4%}")
    print(f"Result: {'REJECT' if p_val < ALPHA else 'FAIL TO REJECT'} H0 at alpha = .05")

    return {"RQ": "RQ4", "method": "independent-samples t-test", "statistic": t_stat,
            "p_value": p_val, "n": len(tech) + len(nontech), "mean_diff": mean_diff}


rq4_result = rq4_test(df)

Levene's test (equal variance): p = 0.6582 -> equal variance assumed
Independent-samples t-test: t = -0.159, p = 0.8740
Mean CAR difference (tech - non-tech): -2.4440%
Result: FAIL TO REJECT H0 at alpha = .05


## Save all results

In [ ]:
results = [rq1_result, rq2_result, rq3_result, rq4_result]
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS_DIR, "rq_results_summary.csv"), index=False)
print(f"All results saved -> {RESULTS_DIR}/rq_results_summary.csv")
results_df

All results saved -> /content/QM640-WALSH-CAPSTONE/results/rq_results_summary.csv


,RQ,method,statistic,p_value,n,CAAR,eta_squared,r_squared,adj_r_squared,mean_diff
0,RQ1,Wilcoxon signed-rank (non-parametric fallback),466.000000,0.064809,51,-0.001632,NaN,NaN,NaN,NaN
1,RQ2,Kruskal-Wallis (non-parametric fallback),0.551577,0.758973,51,NaN,0.001798,NaN,NaN,NaN
2,RQ3,multiple regression with interaction,NaN,NaN,3,NaN,NaN,1.0,NaN,NaN
3,RQ4,independent-samples t-test,-0.159468,0.873956,51,NaN,NaN,NaN,NaN,-0.02444


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "results/rq_results_summary.csv"
!git -C {BASE_DIR} commit -m "Step 6: RQ1-RQ4 statistical test results"
!git -C {BASE_DIR} push

[main 017eb10] Step 6: RQ1-RQ4 statistical test results
 1 file changed, 5 insertions(+)
 create mode 100644 results/rq_results_summary.csv
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 664 bytes | 664.00 KiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   2092594..017eb10  main -> main
